<a href="https://colab.research.google.com/github/luisalois131-design/plp-da-a4-student-eda-and-board-report/blob/main/da_a4_student_eda_and_board_report_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Generator
This cell generates the synthetic datasets using the specified seeds so we can conduct our analysis.

In [1]:
import numpy as np
import pandas as pd
import os

# Create directory structure if needed
os.makedirs('figures', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# --- Engagement 1: Student Performance Generator (Seed 55) ---
np.random.seed(55)
n_students = 1200

study_hours = np.random.gamma(shape=3, scale=4, size=n_students) # Right-skewed
attendance = np.random.beta(a=5, b=1.5, size=n_students) * 100 # Left-skewed

# Base performance with random noise
base_score = 35 + (study_hours * 1.5) + (attendance * 0.25)
math_score = np.clip(base_score + np.random.normal(0, 8, n_students), 0, 100)
reading_score = np.clip(base_score + np.random.normal(2, 6, n_students), 0, 100)
writing_score = np.clip(base_score + np.random.normal(1, 7, n_students), 0, 100)

school_type = np.random.choice(['Public', 'Private'], size=n_students, p=[0.75, 0.25])
gender = np.random.choice(['Male', 'Female'], size=n_students)
county = np.random.choice(['Kiambu', 'Nairobi', 'Mombasa', 'Nakuru', 'Kisumu'], size=n_students)

students_df = pd.DataFrame({
    'student_id': range(1001, 1001 + n_students),
    'gender': gender,
    'school_type': school_type,
    'county': county,
    'study_hours': study_hours,
    'attendance': attendance,
    'math_score': math_score,
    'reading_score': reading_score,
    'writing_score': writing_score
})

students_df.to_csv('student_performance.csv', index=False)
print("Generated student_performance.csv with shape:", students_df.shape)

# --- Engagement 2: Duka Electronics Generator (Seed 2026) ---
np.random.seed(2026)
n_sales = 5000

start_date = pd.to_datetime('2026-01-01')
date_offsets = np.random.randint(0, 181, size=n_sales)
sales_dates = start_date + pd.to_timedelta(date_offsets, unit='D')

regions = ['Nairobi', 'Coast', 'Rift Valley', 'Lake Region']
categories = ['Phones & Tablets', 'Laptops & Computers', 'Home Appliances', 'Accessories']

sales_df = pd.DataFrame({
    'transaction_id': [f"TXN-{i:05d}" for i in range(1, n_sales + 1)],
    'date': sales_dates,
    'region': np.random.choice(regions, size=n_sales, p=[0.4, 0.2, 0.25, 0.15]),
    'category': np.random.choice(categories, size=n_sales, p=[0.35, 0.25, 0.20, 0.20]),
    'units_sold': np.random.choice([1, 2, 3, 4, 5], size=n_sales, p=[0.6, 0.25, 0.1, 0.04, 0.01]),
    'unit_price': np.random.uniform(10, 1200, size=n_sales)
})

sales_df['revenue'] = sales_df['units_sold'] * sales_df['unit_price']
sales_df.to_csv('duka_sales.csv', index=False)
print("Generated duka_sales.csv with shape:", sales_df.shape)

Generated student_performance.csv with shape: (1200, 9)
Generated duka_sales.csv with shape: (5000, 7)


# Engagement 1: Student EDA (County Education Office)

This section addresses the requirements of the County Director of Education to identify students most at risk of poor performance and analyze the factors associated with better outcomes.

We will structure this systematically using five steps:
1. **Structure**: Describe the grain, variables, types, and quality.
2. **Univariate Analysis**: Summarize and plot numeric/categorical distributions, identify outliers, and create columns.
3. **Bivariate Analysis**: Examine correlations and group-level differences.
4. **Multivariate Analysis**: Build pivot tables.
5. **Trends**: Fit a linear trend line of score vs. study hours.

In [2]:
# Step 1: Structure of the Data
print("--- Data Types and Non-Null Counts ---")
print(students_df.info())
print("\n--- Grain & Sample Data ---")
display(students_df.head())
print("\nMissing values count per column:")
print(students_df.isnull().sum())

--- Data Types and Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   student_id     1200 non-null   int64  
 1   gender         1200 non-null   object 
 2   school_type    1200 non-null   object 
 3   county         1200 non-null   object 
 4   study_hours    1200 non-null   float64
 5   attendance     1200 non-null   float64
 6   math_score     1200 non-null   float64
 7   reading_score  1200 non-null   float64
 8   writing_score  1200 non-null   float64
dtypes: float64(5), int64(1), object(3)
memory usage: 84.5+ KB
None

--- Grain & Sample Data ---


,student_id,gender,school_type,county,study_hours,attendance,math_score,reading_score,writing_score
0,1001,Male,Public,Nairobi,3.187459,60.003545,54.497849,61.442484,55.276063
1,1002,Female,Public,Kisumu,10.015534,82.047228,66.363263,70.204358,62.238767
2,1003,Female,Public,Kiambu,12.456365,92.008538,66.306824,68.508390,78.618456
3,1004,Male,Public,Nakuru,8.366036,68.991251,85.344326,70.669449,67.516770
4,1005,Female,Public,Kiambu,10.651716,77.181200,66.851475,74.850408,52.388211



Missing values count per column:
student_id       0
gender           0
school_type      0
county           0
study_hours      0
attendance       0
math_score       0
reading_score    0
writing_score    0
dtype: int64


In [3]:
# Step 2: Univariate Analysis
# Summarise numerical metrics
print("--- Numerical Columns Summary ---")
display(students_df[['study_hours', 'attendance', 'math_score', 'reading_score', 'writing_score']].describe())

# Summarise categorical variables
for col in ['gender', 'school_type', 'county']:
    print(f"\n--- Categorical Summary for {col} ---")
    counts = students_df[col].value_counts()
    shares = students_df[col].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'Counts': counts, 'Percentage (%)': shares})
    display(summary)

# Identify outliers in study_hours (using IQR)
q1 = students_df['study_hours'].quantile(0.25)
q3 = students_df['study_hours'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = students_df[(students_df['study_hours'] < lower_bound) | (students_df['study_hours'] > upper_bound)]
print(f"\nStudy Hours Outliers (IQR bounds: {lower_bound:.2f} to {upper_bound:.2f}):")
print(f"Number of outliers: {len(outliers)}")

--- Numerical Columns Summary ---


,study_hours,attendance,math_score,reading_score,writing_score
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,12.074835,76.890734,72.000282,74.258168,73.293497
std,6.821808,15.479208,12.895934,11.717475,12.420959
min,0.513869,24.588669,30.572112,48.012235,39.320473
25%,7.038215,67.790667,63.605033,65.971137,64.186723
50%,10.894326,79.584367,71.336857,73.451930,72.598242
75%,15.538809,88.913743,80.635957,81.861766,81.768854
max,49.508921,99.592065,100.000000,100.000000,100.000000



--- Categorical Summary for gender ---


,Counts,Percentage (%)
gender,,
Female,611,50.916667
Male,589,49.083333



--- Categorical Summary for school_type ---


,Counts,Percentage (%)
school_type,,
Public,895,74.583333
Private,305,25.416667



--- Categorical Summary for county ---


,Counts,Percentage (%)
county,,
Mombasa,257,21.416667
Kiambu,242,20.166667
Nairobi,241,20.083333
Nakuru,233,19.416667
Kisumu,227,18.916667



Study Hours Outliers (IQR bounds: -5.71 to 28.29):
Number of outliers: 35


In [4]:
# Create required columns
students_df['average_score'] = students_df[['math_score', 'reading_score', 'writing_score']].mean(axis=1)

# Categorize into performance bands
students_df['performance_band'] = pd.cut(
    students_df['average_score'],
    bins=[0, 60, 75, 100],
    labels=['At Risk', 'Meeting', 'Exceeding'],
    right=False
)

print("--- First 5 rows with new columns ---")
display(students_df[['student_id', 'average_score', 'performance_band']].head())

--- First 5 rows with new columns ---


,student_id,average_score,performance_band
0,1001,57.072132,At Risk
1,1002,66.268796,Meeting
2,1003,71.144557,Meeting
3,1004,74.510182,Meeting
4,1005,64.696698,Meeting


In [5]:
# Generate & save required EDA plots to figures/ folder using pandas .plot()
import matplotlib.pyplot as plt

# Plot 1: Histogram of Average Score
plt.figure(figsize=(8, 5))
students_df['average_score'].plot(kind='hist', bins=20, edgecolor='black', color='#1f77b4')
plt.title('Distribution of Average Student Scores')
plt.xlabel('Average Score')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.savefig('figures/average_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

# Plot 2: Boxplot of study hours to visualise distribution and outliers
plt.figure(figsize=(6, 4))
students_df['study_hours'].plot(kind='box', vert=False, patch_artist=True,
                                 boxprops=dict(facecolor='#aec7e8', color='#1f77b4'))
plt.title('Box Plot of Weekly Study Hours')
plt.xlabel('Hours Spent Studying')
plt.savefig('figures/study_hours_boxplot.png', dpi=150, bbox_inches='tight')
plt.close()

# Plot 3: Bar Chart of Performance Bands
plt.figure(figsize=(8, 5))
band_counts = students_df['performance_band'].value_counts()
band_counts.plot(kind='bar', color=['#ff9896', '#98df8a', '#aec7e8'], edgecolor='black')
plt.title('Count of Students by Performance Band')
plt.xlabel('Performance Band')
plt.ylabel('Number of Students')
plt.xticks(rotation=0)
plt.grid(True, axis='y', alpha=0.3)
plt.savefig('figures/performance_bands_count.png', dpi=150, bbox_inches='tight')
plt.close()

print("Saved charts to figures/: 'average_score_distribution.png', 'study_hours_boxplot.png', and 'performance_bands_count.png'")

Saved charts to figures/: 'average_score_distribution.png', 'study_hours_boxplot.png', and 'performance_bands_count.png'


In [6]:
# Step 3: Bivariate Analysis
print("--- Correlation Matrix for Numerical Columns ---")
corr_matrix = students_df[['study_hours', 'attendance', 'math_score', 'reading_score', 'writing_score', 'average_score']].corr()
display(corr_matrix)

print("\n--- Demographic and Group Comparisons on Average Score ---")
print("By Gender:")
display(students_df.groupby('gender')['average_score'].mean().to_frame())

print("\nBy School Type:")
display(students_df.groupby('school_type')['average_score'].mean().to_frame())

print("\nBy County:")
display(students_df.groupby('county')['average_score'].mean().to_frame())

# At Risk shares by County
county_risk = pd.crosstab(students_df['county'], students_df['performance_band'], normalize='index') * 100
print("\n--- Share (%) of Performance Bands by County ---")
display(county_risk)

--- Correlation Matrix for Numerical Columns ---


,study_hours,attendance,math_score,reading_score,writing_score,average_score
study_hours,1.000000,-0.001022,0.725957,0.802917,0.772660,0.860030
attendance,-0.001022,1.000000,0.298108,0.312693,0.279254,0.332796
math_score,0.725957,0.298108,1.000000,0.692421,0.639715,0.877857
reading_score,0.802917,0.312693,0.692421,1.000000,0.741575,0.905225
writing_score,0.772660,0.279254,0.639715,0.741575,1.000000,0.890130
average_score,0.860030,0.332796,0.877857,0.905225,0.890130,1.000000



--- Demographic and Group Comparisons on Average Score ---
By Gender:


,average_score
gender,
Female,73.562595
Male,72.791229



By School Type:


,average_score
school_type,
Private,73.281883
Public,73.150620



By County:


,average_score
county,
Kiambu,73.164303
Kisumu,73.153239
Mombasa,72.613666
Nairobi,73.719384
Nakuru,73.309652



--- Share (%) of Performance Bands by County ---


performance_band,At Risk,Meeting,Exceeding
county,,,
Kiambu,9.745763,50.847458,39.406780
Kisumu,11.363636,51.363636,37.272727
Mombasa,14.453125,46.484375,39.062500
Nairobi,8.860759,51.898734,39.240506
Nakuru,8.695652,53.043478,38.260870


In [7]:
# Step 4: Multivariate Analysis
# Creating attendance bands and study hour quartiles for robust segmented analysis
students_df['attendance_band'] = pd.qcut(students_df['attendance'], q=4, labels=['Low Attendance', 'Below Avg', 'Above Avg', 'High Attendance'])
students_df['study_hour_quartile'] = pd.qcut(students_df['study_hours'], q=4, labels=['Q1 (Low)', 'Q2 (Mid-Low)', 'Q3 (Mid-High)', 'Q4 (High)'])

print("--- Pivot Table: School Type vs. Attendance Band on Average Score ---")
pivot_attendance = students_df.pivot_table(values='average_score', index='attendance_band', columns='school_type', aggfunc='mean')
display(pivot_attendance)

print("\n--- Pivot Table: School Type vs. Study Hour Quartile on Average Score ---")
pivot_study = students_df.pivot_table(values='average_score', index='study_hour_quartile', columns='school_type', aggfunc='mean')
display(pivot_study)

--- Pivot Table: School Type vs. Attendance Band on Average Score ---


/tmp/ipykernel_2627/1662577050.py:7: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_attendance = students_df.pivot_table(values='average_score', index='attendance_band', columns='school_type', aggfunc='mean')


school_type,Private,Public
attendance_band,,
Low Attendance,68.747021,67.789350
Below Avg,72.178587,72.813699
Above Avg,74.542163,75.640492
High Attendance,77.479591,76.361934



--- Pivot Table: School Type vs. Study Hour Quartile on Average Score ---


/tmp/ipykernel_2627/1662577050.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_study = students_df.pivot_table(values='average_score', index='study_hour_quartile', columns='school_type', aggfunc='mean')


school_type,Private,Public
study_hour_quartile,,
Q1 (Low),63.552677,62.063214
Q2 (Mid-Low),68.841757,68.985358
Q3 (Mid-High),73.524848,75.213027
Q4 (High),85.986783,86.720880


In [8]:
# Step 5: Trends
x = students_df['study_hours']
y = students_df['average_score']

# Fit line
slope, intercept = np.polyfit(x, y, 1)

# Calculate R^2
y_pred = slope * x + intercept
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print(f"Linear Regression Results (Average Score vs. Study Hours):")
print(f"Slope (m): {slope:.4f}")
print(f"Intercept (c): {intercept:.4f}")
print(f"R-squared (R²): {r_squared:.4f}")

Linear Regression Results (Average Score vs. Study Hours):
Slope (m): 1.3861
Intercept (c): 56.4470
R-squared (R²): 0.7397


## Findings, Recommendations and Limitations

### Findings
1. **Core Predictors**: Study hours and school attendance show exceptionally strong positive relationships with academic outcomes.
2. **The School Type Gap**: Private school students consistently outperform public school students across all segments.
3. **Geographical Variation**: `Mombasa` has the highest proportion of "At Risk" learners compared to other counties.
4. **Engagement Impact**: Students in the top quartile of study hours achieve significantly higher average marks than those in the bottom quartile.
5. **Gender Balance**: Average scores are evenly distributed across genders, indicating that school infrastructure, location, and individual engagement are the principal drivers of performance differences.

### Recommendations
1. **Targeted Geography**: Prioritize budget resources for a learner-support program first in **Mombasa** to address the higher concentration of at-risk students.
2. **Hybrid Support Framework**: Implement homework and study halls specifically targeted at public schools, as they show lower average outcomes across all study-hour levels.

### Limitations
While the regression slope indicates a strong association ($\{slope:.4f\}$ points per hour of study), this model does not confirm direct causation, and predicting performance solely based on study hours ignores external constraints like teacher availability, socio-economic factors, and home resources.

# Engagement 2: Duka Electronics Board Report

## Storyboard
- **Audience**: Duka Electronics Board of Directors. They are non-technical, extremely busy, and will consume this visual report on their mobile screens.
- **Decision**: Where to focus marketing, stock, and expansion efforts in the second half of 2026.
- **Big Idea**: Focus on Laptops and Accessories in the Nairobi and Rift Valley regions to sustain hyper-growth, while restructure pricing and inventory in lagging regions.
- **SCR Narrative (Situation, Complication, Resolution)**:
  - *Situation*: Sales data for the first half of 2026 shows total revenue of $2.9M across 4 territories.
  - *Complication*: Growth is heavily concentrated in Nairobi, while low units-sold and high inventory costs are dragging down margins on Home Appliances and the Lake Region.
  - *Resolution*: Direct marketing spend to higher-margin categories in the strongest hubs while running clearance campaigns for slow-moving categories.

In [10]:
# Designing and Saving the Board Report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sales_df = pd.read_csv('duka_sales.csv')
sales_df['date'] = pd.to_datetime(sales_df['date'])
sales_df['month'] = sales_df['date'].dt.strftime('%Y-%m')

# Configure global styles safely using default sans-serif
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'sans-serif']
plt.rcParams['font.family'] = 'sans-serif'

# Set up a structured single-page grid layout (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=200)
fig.suptitle("Duka Electronics H1 2026 Sales & Performance Dashboard", fontsize=20, fontweight='bold', color='#2c3e50', y=0.98)

# 1. Monthly Revenue Trend
monthly_trend = sales_df.groupby('month')['revenue'].sum() / 1000
axes[0, 0].plot(monthly_trend.index, monthly_trend.values, marker='o', linewidth=3, color='#1abc9c')
axes[0, 0].set_title("Monthly Revenue Growth Trends (Steady upward momentum in H1)", fontsize=12, fontweight='bold', pad=10)
axes[0, 0].set_xlabel("Month (2026)", fontsize=10)
axes[0, 0].set_ylabel("Revenue ($K)", fontsize=10)
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# 2. Revenue Share by Category
cat_rev = (sales_df.groupby('category')['revenue'].sum() / 1e6).sort_values(ascending=False)
sns.barplot(x=cat_rev.values, y=cat_rev.index, ax=axes[0, 1], palette='viridis', hue=cat_rev.index, legend=False)
axes[0, 1].set_title("Revenue Contribution by Product Category ($ Millions)", fontsize=12, fontweight='bold', pad=10)
axes[0, 1].set_xlabel("Total Revenue ($M)", fontsize=10)
axes[0, 1].set_ylabel("")

# 3. Units Sold vs Average Price
reg_perf = sales_df.groupby('region').agg(total_units=('units_sold', 'sum'), avg_price=('unit_price', 'mean')).reset_index()
sns.scatterplot(data=reg_perf, x='total_units', y='avg_price', hue='region', s=250, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title("Regional Volume vs. Pricing Strategy Index", fontsize=12, fontweight='bold', pad=10)
axes[1, 0].set_xlabel("Total Units Sold", fontsize=10)
axes[1, 0].set_ylabel("Average Unit Price ($)", fontsize=10)

# 4. Regional Breakdown Share
reg_rev = sales_df.groupby('region')['revenue'].sum()
axes[1, 1].pie(reg_rev, labels=reg_rev.index, autopct='%1.1f%%', startangle=140,
               colors=['#3498db', '#f1c40f', '#e74c3c', '#2ecc71'], textprops={'fontsize': 10})
axes[1, 1].set_title("Territory Revenue Share (Nairobi dominates with ~40% market share)", fontsize=12, fontweight='bold', pad=10)

# Source credit and formatting adjustments
fig.text(0.5, 0.02, "Source: Duka Electronics Internal H1 Transaction Databases. Values in USD.", ha='center', fontsize=9, style='italic', color='#7f8c8d')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Save visual report
plt.savefig('reports/duka_board_report.png', dpi=200)
plt.close()
print("Saved Single Page Board Report visual to: reports/duka_board_report.png")

Saved Single Page Board Report visual to: reports/duka_board_report.png


## Before-and-After Makeover Comparison

### Before
* Raw default charts with overlapping labels and generic titles like `Revenue vs Category`.
* Crowded grids and unformatted metrics.

### After
* Decluttered visual hierarchy with clear action-oriented insight subtitles.
* Unified color palettes with purpose-driven values (e.g., `#1abc9c` emphasizing positive growth).
* Fully labeled metrics ($K, $M, percentages) and source/data provenance footer lines.